In [ ]:
from pathlib import Path
import shutil
import sys
import tempfile
import zipfile

import matplotlib.pyplot as plt
import numpy as np
from obspy import read

CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name == "examples" else CWD
REF_EXAMPLES = ROOT.parent / "ref" / "CC-FJpy-master" / "examples"
sys.path.insert(0, str(ROOT))

import ccfj


In [ ]:
def extract_zip(zip_path: Path) -> Path:
    tmp_root = Path(tempfile.mkdtemp(prefix="fjpy_win_eq_nb_"))
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(tmp_root)
    return tmp_root


def load_station_records(data_root: Path):
    sacs = sorted(data_root.glob("*.SAC"))
    if not sacs:
        raise RuntimeError("未找到 SAC 文件。")

    grouped = {}
    for sac in sacs:
        parts = sac.name.split(".")
        if len(parts) < 2:
            continue
        station = ".".join(parts[:2])
        grouped.setdefault(station, []).append(sac)

    stations = sorted(grouped)
    traces = []
    names = []
    for sta in stations:
        tr = read(str(sorted(grouped[sta])[0]))[0]
        tr.detrend("demean")
        traces.append(tr)
        names.append(sta)
    return names, traces


In [ ]:
# 参数恢复为原项目 notebook 的同规模设置，不再做台站数和频散网格简化。
src = REF_EXAMPLES / "eqdata.zip"
tmp_root = extract_zip(src)
names, traces = load_station_records(tmp_root)

nsta = len(traces)
npts = min(tr.stats.npts for tr in traces)
Fs = float(traces[0].stats.sampling_rate)
u0 = np.zeros((nsta, npts), dtype=np.float32)
r = np.zeros(nsta, dtype=np.float32)
for i, tr in enumerate(traces):
    u0[i, :] = tr.data[:npts].astype(np.float32, copy=False)
    r[i] = float(tr.stats.sac["dist"]) if hasattr(tr.stats, "sac") and "dist" in tr.stats.sac else float(i + 1) * 10000.0

nwin = 3
nf = 1000
nc = 1200
minc = 2000.0
maxc = 6000.0
f = Fs * np.linspace(0, nf - 1, nf, dtype=np.float32) / npts
c = np.linspace(minc, maxc, nc, dtype=np.float32)
EQT = 60.0
V1 = [3.2, 3.7]
V2 = [3.7, 4.2]
winl = np.zeros((nwin, nsta), dtype=np.float32)
winr = np.zeros((nwin, nsta), dtype=np.float32)
for i in range(nsta):
    winl[0, i] = 0
    winr[0, i] = npts - 1
    winl[1, i] = r[i] / V2[0] + EQT - 5
    winr[1, i] = r[i] / V1[0] + EQT + 5
    winl[2, i] = r[i] / V2[1] + EQT - 5
    winr[2, i] = r[i] / V1[1] + EQT + 5

indx = np.argsort(r)
u0 = u0[indx, :]
r = r[indx] * 1e3
winl = winl[:, indx]
winr = winr[:, indx]
names = [names[i] for i in indx]

print({
    "stations": names,
    "u0_shape": u0.shape,
    "r_shape": r.shape,
    "f_shape": f.shape,
    "c_shape": c.shape,
})


In [ ]:
out = ccfj.MWFJ(u0, r, c, f, Fs, nwin, winl, winr, taper=0.9, fstride=1, itype=1, func=0, num=4)
out1 = ccfj.MWFJ(u0, r, c, f, Fs, nwin, winl, winr, taper=0.9, fstride=1, itype=1, func=1, num=4)

summary = {
    "stations": names,
    "shape_bessel": list(out.shape),
    "shape_hankel": list(out1.shape),
    "bessel_max": float(np.max(out)),
    "bessel_min": float(np.min(out)),
    "bessel_mean": float(np.mean(out)),
    "bessel_std": float(np.std(out)),
    "bessel_window_mean": [float(np.mean(out[i])) for i in range(out.shape[0])],
    "hankel_max": float(np.max(out1)),
    "hankel_min": float(np.min(out1)),
    "hankel_mean": float(np.mean(out1)),
    "hankel_std": float(np.std(out1)),
    "hankel_window_mean": [float(np.mean(out1[i])) for i in range(out1.shape[0])],
}
summary


In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(18, 10), constrained_layout=True)
for i in range(3):
    ax[0][i].imshow(
        np.flip(out[i, :, :], 0),
        extent=[float(np.min(f)), float(np.max(f)), float(np.min(c / 1e3)), float(np.max(c / 1e3))],
        aspect="auto",
        vmax=0.8,
        cmap="jet",
    )
    ax[0][i].set_title(f"Window {i + 1} - Bessel")
    ax[0][i].set_xlabel("Frequency (Hz)")
    ax[0][i].set_ylabel("Phase velocity (km/s)")
    ax[1][i].imshow(
        np.flip(out1[i, :, :], 0),
        extent=[float(np.min(f)), float(np.max(f)), float(np.min(c / 1e3)), float(np.max(c / 1e3))],
        aspect="auto",
        vmax=0.8,
        cmap="jet",
    )
    ax[1][i].set_title(f"Window {i + 1} - Hankel")
    ax[1][i].set_xlabel("Frequency (Hz)")
    ax[1][i].set_ylabel("Phase velocity (km/s)")
plt.show()


In [ ]:
shutil.rmtree(tmp_root, ignore_errors=True)
print("中文自检：地震 F-J notebook 已按 UTF-8 写入。")
